# NB00 — Ingest & consolidate

**Input:** `data/processed/*_panel.parquet` (60 per-company panels, already ingested)

**Output:** `panel_long.parquet`, `macro_q.parquet`, a missing-value ledger, and a W&B run `nb00-ingest-panel`

**Spec:** `docs/specs/nb00-ingest-and-consolidate.md` · **Plan:** `docs/plans/nb00-ingest-and-consolidate.md`

_This notebook never runs ingestion._

## E1 — is the 2026-Q2 row reported or projected?

**Answered 2026-09-23: reported.** 2026-Q2 (date 2026-06-30) is the last reported quarter and is the forecast origin. 2026-Q3 ends 2026-09-30, has not been reported, and is horizon 1. Panel evidence agrees: all 60 companies have 2026-Q2 revenue and none repeats the prior quarter. `Settings.LAST_REPORTED_QUARTER` holds this as data; bump it when a newer quarter is ingested.

**Known limitation — fiscal calendars.** Ingestion snaps each fiscal period end to the nearest calendar quarter end within 46 days, so companies with off-calendar fiscal years (AAPL, MSFT, PG, NKE, COST, DE, WMT, HD, M, BBY, ORCL, CSCO, ADBE, …) line up with macro data only within that margin. Not corrected here.

## 1. Load panels

Validated read of the 60 per-company files. **Exit:** 60 frames × 81 rows.

In [1]:
from app.data import write_panel
from app.data.consolidation import consolidate_panels
from app.data.flags import add_flags
from app.data.missing import missing_value_ledger
from app.injections import configure_container
from app.services.tracking import RunConfig, run_name
from app.settings import Settings

container = configure_container()
tracker = container.experiment_tracker()
panels = container.panel_store().load_all()

EXPECTED_COMPANIES = 60
EXPECTED_QUARTERS = 81
EXPECTED_COVID_ROWS = 3 * EXPECTED_COMPANIES
EXPECTED_BREAK_ROWS = 7 * 4  # 7 breaks x 4 quarters; changes if the GE entries do

n_rows = sum(len(panel) for panel in panels.values())
assert len(panels) == EXPECTED_COMPANIES  # A1
assert all(len(panel) == EXPECTED_QUARTERS for panel in panels.values())

2026-09-24 22:35:16.872 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 22:35:16.873 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## 2. Consolidate and flag

`panel_long` (no macro, no `is_public`) + `macro_q` (81 rows); flags `covid`, `structural_break`, `outlier_flag`, `is_projected`. **Exit:** row count = sum of per-company rows (4,860); macro table exactly 81 rows.

In [2]:
consolidated = consolidate_panels(panels)
macro_q = consolidated.macro_q
panel_long = add_flags(
    consolidated.panel_long,
    container.structural_breaks(),
    last_reported_quarter=Settings.LAST_REPORTED_QUARTER,
)

assert len(panel_long) == n_rows
assert len(macro_q) == EXPECTED_QUARTERS  # A2
assert "is_public" not in panel_long.columns
assert not panel_long["is_projected"].any()  # A3
assert panel_long["covid"].sum() == EXPECTED_COVID_ROWS  # A4
assert panel_long["structural_break"].sum() == EXPECTED_BREAK_ROWS  # A5
amzn_latest = panel_long.query("ticker == 'AMZN' and date == '2026-06-30'")
assert amzn_latest["outlier_flag"].item()  # A6

C:\Users\leona\AppData\Local\Temp\ipykernel_3220\386250155.py:15: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  amzn_latest = panel_long.query("ticker == 'AMZN' and date == '2026-06-30'")


## 3. Missing-value ledger

Every NaN cell with a reason. **Exit:** each `unexplained` row is fixed by re-ingestion or waived here in writing (expected today: COST EPS ×9, COST EBITDA 2026-06-30).

In [3]:
ledger = missing_value_ledger(panel_long)
ledger.groupby(["column", "reason"]).size()  # A7
ledger.query("reason == 'unexplained'")

,ticker,date,column,reason
98,COST,2026-06-30,ebitda_usd_m,unexplained
99,COST,2008-06-30,eps,unexplained
100,COST,2009-06-30,eps,unexplained
101,COST,2010-06-30,eps,unexplained
102,COST,2011-06-30,eps,unexplained
103,COST,2012-06-30,eps,unexplained
104,COST,2013-06-30,eps,unexplained
105,COST,2016-06-30,eps,unexplained
106,COST,2017-06-30,eps,unexplained
107,COST,2020-06-30,eps,unexplained


**Unexplained rows: resolution.** Fill in after the first run, one line per row
(fixed by re-ingestion, or waived and why). Expected today: COST EPS (9 quarters)
and COST EBITDA (2026-06-30).

## 4. Save and track

Write both parquet files and log them as W&B dataset artifacts `panel_long` and `macro_q`.

In [4]:
config = RunConfig(panel_size=len(panels), n_rows=n_rows)
flag_counts = (
    panel_long[["covid", "structural_break", "outlier_flag", "is_projected"]]
    .sum()
    .rename("rows")
    .rename_axis("flag")
    .reset_index()
)
with tracker.start_run(
    run_name(notebook="nb00", model="ingest", variable="panel"),
    config,
    job_type="consolidate",
) as run:
    run.log_table("missing_value_ledger", ledger)
    run.log_table("flag_counts", flag_counts)
    run.log_dataset("panel_long", write_panel(panel_long, Settings.PANEL_LONG_PATH))
    run.log_dataset("macro_q", write_panel(macro_q, Settings.MACRO_Q_PATH))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\leona\_netrc
wandb: Currently logged in as: leonardo-heis (leonardo-a-heis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.
